##### Step1 Filter the interaction data from NPInter and separately obtain the LPI for human and mouse.

In [2]:
import pandas as pd

input_file = "../../data/raw/lncRNA_interaction.txt"
output_human = "human_npinter_lmi.csv"
output_mouse = "mouse_npinter_lmi.csv"

# indices to keep: (gene_name, gene_id, protein_name, uniprot_id, tissue_or_cellline)
keep_indices = [1, 2, 4, 5, 11]

# final output header
header = ['gene_name', 'gene_id', 'identifier', 'miRNA', 'miRNA_id', 'tissue_or_cellline']

# read input file
df = pd.read_csv(input_file, sep="\t", header=None, dtype=str)

# filter rows: lncRNA - miRNA - binding
df = df[
    (df[3] == "lncRNA") &
    (df[6] == "miRNA") &
    (df[13].astype(str).str.contains("binding", case=False, na=False))
]


# fix gene_id
df.iloc[:, 2] = df.iloc[:, 2].replace("NOCODE", "-")

# select needed columns
filtered = df.iloc[:, keep_indices].copy()
filtered.columns = ['gene_name', 'gene_id', 'miRNA', 'miRNA_id', 'tissue_or_cellline']

# construct identifier column
filtered["identifier"] = filtered.apply(
    lambda x: x["gene_id"] if x["gene_id"] != "-" else x["gene_name"], axis=1
)

# reorder columns
filtered = filtered[['gene_name', 'gene_id', 'identifier', 'miRNA', 'miRNA_id']]

# remove self-loops (gene_name == miRNA)
filtered = filtered[~((filtered['gene_name'] != "") & (filtered['miRNA'] != "") & 
                     (filtered['gene_name'] == filtered['miRNA']))]

# split by species
human_lmi = filtered[df[10] == "Homo sapiens"].drop_duplicates()
mouse_lmi = filtered[df[10] == "Mus musculus"].drop_duplicates()
# save to CSV
human_lmi.to_csv(output_human, index=False, encoding='utf-8')
mouse_lmi.to_csv(output_mouse, index=False, encoding='utf-8')


##### Step2：Fix the LPI data.

Step 2.1 Replace the transcript IDs with their corresponding gene IDs.

In [3]:
# Human
# -----------------------------
# Paths
# -----------------------------
mapping_file_noncode6 = "../../reference_lncRNA/human/transcript/NONCODEv6_human_hg38_lncRNA_trans.txt"
mapping_file_noncode5 = "../../reference_lncRNA/human/transcript/NONCODEv5_human_hg38_lncRNA_trans.txt"
out_file = "human_lmi_id_fixed.csv"

# Transcript-like ID prefixes to repair
transcript_prefixes = ("NONHSAT",)

# -----------------------------
# Helper: load transcript->gene mapping
# Assumes each line has at least two columns:
#   col0 = gene_id, col1 = transcript_id
# Auto-detects delimiter (comma/tab/whitespace) via engine='python'.
# -----------------------------
def load_mapping(path: str) -> dict:
    df = pd.read_csv(
        path,
        sep=None,                # auto-detect delimiter
        engine="python",
        header=None,
        usecols=[0, 1],          # [gene_id, transcript_id]
        names=["gene_id", "transcript_id"],
        dtype=str
    )
    # normalize strings
    df = df.dropna(subset=["gene_id", "transcript_id"])
    df["gene_id"] = df["gene_id"].str.strip()
    df["transcript_id"] = df["transcript_id"].str.strip()
    # build transcript -> gene mapping (v6/v5 priority handled outside)
    return dict(zip(df["transcript_id"], df["gene_id"]))

# Load mappings: v6 has higher priority than v5
map_v6 = load_mapping(mapping_file_noncode6)
map_v5 = load_mapping(mapping_file_noncode5)

# -----------------------------
# Load lmi table
# Keep strings to avoid unintended type casting
# -----------------------------

# Normalize gene_id string for testing and lookups
gid_norm = human_lmi["gene_id"].fillna("").astype(str).str.strip()

# Identify rows that look like transcript IDs (to be repaired)
mask_tx_like = gid_norm.str.startswith(transcript_prefixes)

# Map transcript -> gene via v6 (priority) and v5 (fallback) only on those rows
mapped_v6 = gid_norm.where(mask_tx_like).map(map_v6)
mapped_v5 = gid_norm.where(mask_tx_like).map(map_v5)

# Start with original gene_id and apply repairs
new_gene_id = human_lmi["gene_id"].copy()

# Apply v6 where available
mask_v6_hit = mapped_v6.notna()
new_gene_id.loc[mask_v6_hit] = mapped_v6.loc[mask_v6_hit].values

# Apply v5 where v6 missed but v5 hit
mask_v5_hit = (~mask_v6_hit) & mapped_v5.notna()
new_gene_id.loc[mask_v5_hit] = mapped_v5.loc[mask_v5_hit].values

# Update gene_id in the dataframe
human_lmi["gene_id"] = new_gene_id

# Rows actually repaired (either v6 or v5 hit)
mask_repaired = mask_v6_hit | mask_v5_hit

# Keep identifier in sync with repaired gene_id (same behavior as original script)
human_lmi.loc[mask_repaired, "identifier"] = human_lmi.loc[mask_repaired, "gene_id"]

# Save
human_lmi.to_csv(out_file, index=False)


In [4]:
# Mouse
# -----------------------------
# Paths
# -----------------------------
mapping_file_noncode5 = "../../reference_lncRNA/mouse/transcript/NONCODEv5_mouse_mm10_lncRNA_trans.txt"
out_file = "mouse_lmi_id_fixed.csv"

transcript_prefixes = ("NONMMUT",)

# -----------------------------
# Helper: load transcript->gene mapping
# Assumes each line has at least two columns:
#   col0 = gene_id, col1 = transcript_id
# Auto-detects delimiter (comma/tab/whitespace) via engine='python'.
# -----------------------------
def load_mapping(path: str) -> dict:
    df = pd.read_csv(
        path,
        sep=None,                # auto-detect delimiter
        engine="python",
        header=None,
        usecols=[0, 1],          # [gene_id, transcript_id]
        names=["gene_id", "transcript_id"],
        dtype=str
    )
    # normalize strings
    df = df.dropna(subset=["gene_id", "transcript_id"])
    df["gene_id"] = df["gene_id"].str.strip()
    df["transcript_id"] = df["transcript_id"].str.strip()
    # build transcript -> gene mapping (v6/v5 priority handled outside)
    return dict(zip(df["transcript_id"], df["gene_id"]))

# Load mappings
map_v5 = load_mapping(mapping_file_noncode5)

# -----------------------------
# Load lmi table
# Keep strings to avoid unintended type casting
# -----------------------------

# Normalize gene_id string for testing and lookups
gid_norm = mouse_lmi["gene_id"].fillna("").astype(str).str.strip()

# Identify rows that look like transcript IDs (to be repaired)
mask_tx_like = gid_norm.str.startswith(transcript_prefixes)

# Map transcript -> gene via v5
mapped_v5 = gid_norm.where(mask_tx_like).map(map_v5)

# Start with original gene_id and apply repairs
new_gene_id = mouse_lmi["gene_id"].copy()

# Apply v5
mask_v5_hit = mapped_v5.notna()
new_gene_id.loc[mask_v5_hit] = mapped_v5.loc[mask_v5_hit].values

# Update gene_id in the dataframe
mouse_lmi["gene_id"] = new_gene_id

# Rows actually repaired
mask_repaired = mask_v5_hit

# Keep identifier in sync with repaired gene_id (same behavior as original script)
mouse_lmi.loc[mask_repaired, "identifier"] = mouse_lmi.loc[mask_repaired, "gene_id"]

# Save
mouse_lmi.to_csv(out_file, index=False)


Step2.2 Replace invalid identifier with gene_name

In [5]:
# Human
import pandas as pd
import os
import re

# ----------------------------
# Paths (adjust as needed)
# ----------------------------
ensembl_dir = "../../reference_lncRNA/human/bed/ensembl/"
# human_lmi = pd.read_csv('human_lmi_id_fixed.csv')

# ----------------------------
# Load NONCODE gene_id lists (only gene_id column is needed)
# NONCODE BED columns: chr, start, end, gene_id, score, strand
# ----------------------------
noncodev5_ids = pd.read_csv(
    '../../reference_lncRNA/human/bed/NONCODEv5_hg38.lncRNAGene.bed',
    sep='\t', header=None, usecols=[3], names=['gene_id']
)['gene_id']

noncodev6_ids = pd.read_csv(
    '../../reference_lncRNA/human/bed/NONCODEv6_hg38.lncRNAGene.bed',
    sep='\t', header=None, usecols=[3], names=['gene_id']
)['gene_id']

# ----------------------------
# Collect Ensembl gene_id from all BED files in the directory
# Ensembl BED columns: chr, start, end, gene_name, gene_id, strand
# ----------------------------
def extract_version(filename: str) -> int:
    """
    Extract Ensembl GRCh38 version number from filename (e.g., '...GRCh38.<n>.bed').
    Returns -1 if not matched.
    """
    m = re.search(r'GRCh38\.(\d+)\.bed', filename)
    return int(m.group(1)) if m else -1

bed_files = [f for f in os.listdir(ensembl_dir) if f.endswith(".bed")]
# Sorting not strictly required for validity checking, but kept for consistency
bed_files_sorted = sorted(bed_files, key=extract_version, reverse=True)

ensembl_ids_list = []
for bed_file in bed_files_sorted:
    bed_path = os.path.join(ensembl_dir, bed_file)
    # Read only the gene_id column (index 4)
    gid = pd.read_csv(bed_path, sep='\t', header=None, usecols=[4], names=['gene_id'])['gene_id']
    ensembl_ids_list.append(gid)

ensembl_ids = pd.concat(ensembl_ids_list, ignore_index=True) if ensembl_ids_list else pd.Series([], dtype=object)

# ----------------------------
# Build a set of valid gene IDs across all sources
# ----------------------------
def to_id_set(series: pd.Series) -> set:
    """
    Normalize a Series to a set of non-empty string IDs:
    - drop NaN
    - strip spaces
    - drop empty strings
    """
    s = series.dropna().astype(str).str.strip()
    s = s[s != ""]
    return set(s)

valid_ids = to_id_set(noncodev6_ids) | to_id_set(noncodev5_ids) | to_id_set(ensembl_ids)

# ----------------------------
# Determine invalid IDs and update 'identifier' accordingly
# Rules:
# - If gene_id is NOT in valid_ids (or is null/empty) -> treat as invalid
# - For invalid rows: identifier := gene_name (gene_name!=-,gene_name is valid)
# - For valid rows: identifier remains unchanged
# - Filter out rows with invalid gene_id and invalid gene_name
# ----------------------------
# ----------------------------
# Determine valid and invalid gene_id
# ----------------------------
gene_id_raw = human_lmi['gene_id']
gene_id_norm = gene_id_raw.astype(str).str.strip()

mask_valid_id = gene_id_norm.isin(valid_ids)
mask_invalid_id = gene_id_raw.isna() | (gene_id_norm == "") | (~gene_id_norm.isin(valid_ids))

# ----------------------------
# For invalid gene_id, check gene_name validity
# ----------------------------
gene_name_norm = human_lmi['gene_name'].astype(str).str.strip()
mask_valid_name = (gene_name_norm != "") & (gene_name_norm != "-")

# Rows to replace identifier with gene_name
mask_replace = mask_invalid_id & mask_valid_name

# Rows to drop: both ID invalid and name invalid
mask_drop = mask_invalid_id & (~mask_valid_name)

# Apply replacement
human_lmi.loc[mask_replace, 'identifier'] = human_lmi.loc[mask_replace, 'gene_name']

# Drop completely invalid rows
human_lmi = human_lmi.loc[~mask_drop].copy()

# save the updated table
human_lmi.to_csv('./human_lmi_fixed.csv', index=False)


In [6]:
# Mouse
import pandas as pd
import os
import re

# ----------------------------
# Paths (adjust as needed)
# ----------------------------
ensembl_dir = "../../reference_lncRNA/mouse/bed/ensembl/"
#mouse_lmi = pd.read_csv('mouse_lmi_id_fixed.csv')

# ----------------------------
# Load NONCODE gene_id lists (only gene_id column is needed)
# NONCODE BED columns: chr, start, end, gene_id, score, strand
# ----------------------------
noncodev5_ids = pd.read_csv(
    '../../reference_lncRNA/mouse/bed/NONCODEv5_mm10.lncRNAGene.bed',
    sep='\t', header=None, usecols=[3], names=['gene_id']
)['gene_id']

noncodev6_ids = pd.read_csv(
    '../../reference_lncRNA/mouse/bed/NONCODEv6_mm10.lncRNAGene.bed',
    sep='\t', header=None, usecols=[3], names=['gene_id']
)['gene_id']

# ----------------------------
# Collect Ensembl gene_id from all BED files in the directory
# Ensembl BED columns: chr, start, end, gene_name, gene_id, strand
# ----------------------------
def extract_version(filename: str) -> int:
    """
    Extract Ensembl GRCm38 version number from filename (e.g., '...GRCm38.<n>.bed').
    Returns -1 if not matched.
    """
    m = re.search(r'GRCm38\.(\d+)\.bed', filename)
    return int(m.group(1)) if m else -1

bed_files = [f for f in os.listdir(ensembl_dir) if re.match(r"Mus_musculus\.GRCm38\.\d+\.bed$", f)]
# Sorting not strictly required for validity checking, but kept for consistency
bed_files_sorted = sorted(bed_files, key=extract_version, reverse=True)

ensembl_ids_list = []
for bed_file in bed_files_sorted:
    bed_path = os.path.join(ensembl_dir, bed_file)
    # Read only the gene_id column (index 4)
    gid = pd.read_csv(bed_path, sep='\t', header=None, usecols=[4], names=['gene_id'])['gene_id']
    ensembl_ids_list.append(gid)

ensembl_ids = pd.concat(ensembl_ids_list, ignore_index=True) if ensembl_ids_list else pd.Series([], dtype=object)

# ----------------------------
# Build a set of valid gene IDs across all sources
# ----------------------------
def to_id_set(series: pd.Series) -> set:
    """
    Normalize a Series to a set of non-empty string IDs:
    - drop NaN
    - strip spaces
    - drop empty strings
    """
    s = series.dropna().astype(str).str.strip()
    s = s[s != ""]
    return set(s)

valid_ids = to_id_set(noncodev6_ids) | to_id_set(noncodev5_ids) | to_id_set(ensembl_ids)

# ----------------------------
# Determine invalid IDs and update 'identifier' accordingly
# Rules:
# - If gene_id is NOT in valid_ids (or is null/empty) -> treat as invalid
# - For invalid rows: identifier := gene_name (gene_name!=-,gene_name is valid)
# - For valid rows: identifier remains unchanged
# - Filter out rows with invalid gene_id and invalid gene_name
# ----------------------------
# ----------------------------
# Determine valid and invalid gene_id
# ----------------------------
gene_id_raw = mouse_lmi['gene_id']
gene_id_norm = gene_id_raw.astype(str).str.strip()

mask_valid_id = gene_id_norm.isin(valid_ids)
mask_invalid_id = gene_id_raw.isna() | (gene_id_norm == "") | (~gene_id_norm.isin(valid_ids))

# ----------------------------
# For invalid gene_id, check gene_name validity
# ----------------------------
gene_name_norm = mouse_lmi['gene_name'].astype(str).str.strip()
mask_valid_name = (gene_name_norm != "") & (gene_name_norm != "-")

# Rows to replace identifier with gene_name
mask_replace = mask_invalid_id & mask_valid_name

# Rows to drop: both ID invalid and name invalid
mask_drop = mask_invalid_id & (~mask_valid_name)

# Apply replacement
mouse_lmi.loc[mask_replace, 'identifier'] = mouse_lmi.loc[mask_replace, 'gene_name']

# save the updated table
mouse_lmi.to_csv('./mouse_lmi_fixed.csv', index=False)

#### Step 3 Construct LMI networks.

In [4]:
# Human
import pandas as pd

human_lmi = pd.read_csv("human_lmi_fixed.csv")

human_lmi["gene_name"] = (
    human_lmi["gene_name"]
    .astype(str)
    .str.replace("‐", "-", regex=False)
)

lncRNA = pd.read_csv("../../data/LPI/human/lncRNA_mapping.csv")
human_lmi = pd.merge(human_lmi, lncRNA, right_on='member_id', left_on='identifier', how='inner')
human_lmi = human_lmi.drop(['gene_id', 'gene_name', 'member_id', 'identifier', 'miRNA'], axis=1)

human_lmi = human_lmi[human_lmi['miRNA_id']!='-']

human_lmi = human_lmi.drop_duplicates()


human_lmi.to_csv("human_lmi.csv",index=False)


In [5]:
# Mouse
import pandas as pd

mouse_lmi = pd.read_csv("mouse_lmi_fixed.csv")

mouse_lmi["gene_name"] = (
    mouse_lmi["gene_name"]
    .astype(str)
    .str.replace("‐", "-", regex=False)
)

lncRNA = pd.read_csv("../../data/LPI/mouse/lncRNA_mapping.csv")
mouse_lmi = pd.merge(mouse_lmi, lncRNA, right_on='member_id', left_on='identifier', how='inner')
mouse_lmi = mouse_lmi.drop(['gene_id', 'gene_name', 'member_id', 'identifier','miRNA'], axis=1)

mouse_lmi = mouse_lmi[mouse_lmi['miRNA_id']!='-']

mouse_lmi = mouse_lmi.drop_duplicates();
mouse_lmi.to_csv("mouse_lmi.csv",index=False)
